# Pipeline MMM Randstad — esecuzione su Colab

Fit Meridian con GPU. **Runtime → Cambia tipo di runtime → GPU (T4)** prima di iniziare.
Ordine: setup → generator → ingestion (con conferma umana) → fit → recovery → allocator.


In [ ]:
!git clone https://github.com/Giacomod2001/Tesi-MMM.git
%cd Tesi-MMM
!pip install -q -r pipeline/requirements.txt


## 1. Dataset sintetico (export sporchi + ground truth)


In [ ]:
!python -m pipeline.generator.run


## 2. Ingestion — proposta di mappatura

La cella seguente mostra la mappatura proposta per ogni file. **Esaminala**: è il punto human-in-the-middle.


In [ ]:
from pipeline.ingestion import build
from pipeline import config
plans, tables = build.propose_plan(config.RAW_DIR)
for p in plans:
    print(f'\n{p.file}  [{p.kind}' + (f' | {p.channel}' if p.channel else '') + ']')
    for c in p.columns:
        print(f'   {c.source!r:<38} -> {c.field}')
    for n in p.notes: print('   nota:', n)


### Conferma

Se la mappatura è corretta, esegui la cella sotto (equivale alla conferma interattiva).
Per correggere un campo: `plans[i].columns = [...]` prima di confermare.


In [ ]:
for p in plans:
    p.confirmed = True   # conferma esplicita dell'operatore
res = build.ingest(config.RAW_DIR, plan=plans, interactive=False, tables=tables)
for line in res['log']: print(' ', line)


## 3. Fit Meridian (GPU, ~20–40 min)

Per una prova rapida di meccanica usa `--smoke` (stime NON utilizzabili).


In [ ]:
!python -m pipeline.model.run


## 4. Parameter recovery — stima vs verità nascosta


In [ ]:
!python -m pipeline.validation.recovery


## 5. Allocazione trimestrale (esempio con vincoli)


In [ ]:
!python -m pipeline.allocator.run --budget 450000 \
    --min linkedin=50000 --max meta=250000 --quarter-start 2026-07-06


## 6. Scarica gli output


In [ ]:
from google.colab import files
import shutil
shutil.make_archive('output_mmm', 'zip', 'pipeline/data/output')
files.download('output_mmm.zip')
